# Environment Setup

Installing PyTorch, OpenCV, the COCO API, and clone the official YOLOv5
repository. YOLOv5 already implements CIoU loss, mosaic/HSV/flip augmentation, and cosine LR scheduling internally configures.

In [2]:
# --- Install core libraries and COCO API
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 2>/dev/null || pip install -q torch torchvision torchaudio
!pip install -q opencv-python-headless pandas pyyaml tqdm matplotlib seaborn Pillow requests
!pip install -q pycocotools
!pip install -q onnx onnxruntime
!pip install -q tabulate

# Clone YOLOv5
import os

if not os.path.isdir("yolov5"):
    !git clone -q https://github.com/ultralytics/yolov5.git

%cd yolov5
!pip install -q -r requirements.txt
%cd ..

import torch
print("Torch version   :", torch.__version__)
print("CUDA available   :", torch.cuda.is_available())
print("Device            :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 84.4 MB/s eta 0:00:00
/content/yolov5
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 13.6 MB/s eta 0:00:00
/content
Torch version   : 2.11.0+cu128
CUDA available   : True
Device            : Tesla T4


In [3]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Training YOLOv5 variants on CPU is extremely slow "
        "(potentially many hours per variant) and is not recommended for this notebook.\n"
        "In Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4 is fine), "
        "then Runtime -> Restart session, and re-run this notebook from the top."
    )

print(f"GPU detected: {torch.cuda.get_device_name(0)} -- safe to proceed with training.")


GPU detected: Tesla T4 -- safe to proceed with training.


## 1. Data Preprocessing
### Requirement 1: Download and preprocess COCO (resizing + annotation parsing), split 80/10/10
### Tool: COCO API (`pycocotools`)

In [4]:
# 1.1 Download COCO 2017 annotation files (small, ~241MB)
import os

ANNOT_DIR = "coco_annotations"
os.makedirs(ANNOT_DIR, exist_ok=True)

ANNOT_ZIP_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
zip_path = os.path.join(ANNOT_DIR, "annotations_trainval2017.zip")

if not os.path.exists(zip_path):
    !wget -q -O {zip_path} {ANNOT_ZIP_URL}
!unzip -q -o {zip_path} -d {ANNOT_DIR}

print("Annotation files:", os.listdir(os.path.join(ANNOT_DIR, "annotations")))


Annotation files: ['captions_val2017.json', 'instances_train2017.json', 'instances_val2017.json', 'captions_train2017.json', 'person_keypoints_train2017.json', 'person_keypoints_val2017.json']


In [5]:
# 1.2 Load annotations with the official COCO API (pycocotools)
from pycocotools.coco import COCO

train_ann_path = os.path.join(ANNOT_DIR, "annotations", "instances_train2017.json")
val_ann_path   = os.path.join(ANNOT_DIR, "annotations", "instances_val2017.json")

coco_train = COCO(train_ann_path)
coco_val   = COCO(val_ann_path)

categories = coco_train.loadCats(coco_train.getCatIds())      # COCO API call
categories = sorted(categories, key=lambda c: c["id"])
COCO_CLASSES   = [c["name"] for c in categories]
COCO_ID_TO_YOLO_ID = {c["id"]: i for i, c in enumerate(categories)}

print(f"Loaded {len(COCO_CLASSES)} COCO categories via the COCO API.")
print(COCO_CLASSES[:10], "...")


loading annotations into memory...
Done (t=18.64s)
creating index...
index created!
loading annotations into memory...
Done (t=0.51s)
creating index...
index created!
Loaded 80 COCO categories via the COCO API.
['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light'] ...


In [6]:
# 1.3 Stratified sampling: pull a fixed number of images PER CATEGORY (not pure random)
import random
from collections import Counter

random.seed(42)
# images per category; 50 x 80 categories = up to ~4,000 images
IMAGES_PER_CATEGORY = 50

selected = {}  # key: (split_name, img_id)  -> value: coco_api handle (coco_train or coco_val)

for cat in categories:   # loop over all 80 categories; each getImgIds() call is fast (metadata only, no downloads)
    cat_id = cat["id"]

    # training-
    train_ids_for_cat = coco_train.getImgIds(catIds=[cat_id])   # COCO API: images containing this category
    random.shuffle(train_ids_for_cat)

    added = 0
    for img_id in train_ids_for_cat:
        key = ("train2017", img_id)
        if key not in selected:
            selected[key] = coco_train
            added += 1
        if added >= IMAGES_PER_CATEGORY:
            break

    # if train2017 didn't have enough images for this (rare) category, top up from val2017
    if added < IMAGES_PER_CATEGORY:
        val_ids_for_cat = coco_val.getImgIds(catIds=[cat_id])   # COCO API: images containing this category
        random.shuffle(val_ids_for_cat)
        for img_id in val_ids_for_cat:
            key = ("val2017", img_id)
            if key not in selected:
                selected[key] = coco_val
                added += 1
            if added >= IMAGES_PER_CATEGORY:
                break

sampled_ids = [(split, img_id, api) for (split, img_id), api in selected.items()]

print(f"Target: up to {IMAGES_PER_CATEGORY} images/category x {len(categories)} categories "
      f"= {IMAGES_PER_CATEGORY * len(categories)} max.")
print(f"Actual unique images selected: {len(sampled_ids)} "
      f"(lower than the max is expected/fine -- many images contain MULTIPLE categories at once, "
      f"so the same image can count towards several categories' quotas simultaneously).")


Target: up to 50 images/category x 80 categories = 4000 max.
Actual unique images selected: 4000 (lower than the max is expected/fine -- many images contain MULTIPLE categories at once, so the same image can count towards several categories' quotas simultaneously).


In [7]:
# 1.3b Verify category coverage BEFORE downloading anything
import pandas as pd

category_coverage = Counter()
for split_name, img_id, api in sampled_ids:
    ann_ids = api.getAnnIds(imgIds=img_id)
    anns = api.loadAnns(ann_ids)
    cats_in_image = set(a["category_id"] for a in anns)
    for c in cats_in_image:
        category_coverage[c] += 1

coverage_rows = [
    {"category": cat["name"], "images_containing_it": category_coverage.get(cat["id"], 0)}
    for cat in categories
]
coverage_df = pd.DataFrame(coverage_rows).sort_values("images_containing_it").reset_index(drop=True)

n_zero_coverage = (coverage_df["images_containing_it"] == 0).sum()
print(f"Categories with ZERO coverage in our sample: {n_zero_coverage} / {len(categories)}")
print("\nLowest-coverage categories (worth checking):")
print(coverage_df.head(10).to_string(index=False))
print("\nHighest-coverage categories:")
print(coverage_df.tail(5).to_string(index=False))


Categories with ZERO coverage in our sample: 0 / 80

Lowest-coverage categories (worth checking):
     category  images_containing_it
   hair drier                    53
         bear                    54
        zebra                    54
     elephant                    58
        sheep                    59
      giraffe                    61
      toaster                    62
    snowboard                    63
     airplane                    64
parking meter                    65

Highest-coverage categories:
    category  images_containing_it
      bottle                   492
         cup                   573
       chair                   645
dining table                   652
      person                  2110


In [8]:
# 1.4 Download the sampled images from the official COCO image server
import requests
from tqdm import tqdm

RAW_IMG_DIR = "coco_images_raw"
os.makedirs(RAW_IMG_DIR, exist_ok=True)

downloaded = []  # (local_image_path, coco_api_handle, image_id)

for split_name, img_id, coco_api in tqdm(sampled_ids, desc="Downloading COCO images"):
    img_info = coco_api.loadImgs(img_id)[0]      # COCO API: fetch image metadata (incl. URL)
    file_name = f"{split_name}_{img_info['file_name']}"
    out_path = os.path.join(RAW_IMG_DIR, file_name)

    if not os.path.exists(out_path):
        try:
            resp = requests.get(img_info["coco_url"], timeout=10)
            if resp.status_code == 200:
                with open(out_path, "wb") as f:
                    f.write(resp.content)
        except requests.RequestException:
            continue

    if os.path.exists(out_path):
        downloaded.append((out_path, coco_api, img_id))

print(f"Successfully downloaded {len(downloaded)} / {len(sampled_ids)} images.")


Successfully downloaded 4000 / 4000 images.


In [9]:
# 1.5 Parse annotations (via COCO API) + resize images + write YOLO-format labels
import cv2
import numpy as np

IMG_SIZE = 640
STAGE_DIR = "coco_yolo_staging"
os.makedirs(os.path.join(STAGE_DIR, "images"), exist_ok=True)
os.makedirs(os.path.join(STAGE_DIR, "labels"), exist_ok=True)

def letterbox_resize(image, size=IMG_SIZE):
    """Resize keeping aspect ratio, pad the rest (standard YOLO preprocessing)."""
    h, w = image.shape[:2]
    scale = size / max(h, w)
    nh, nw = int(h * scale), int(w * scale)
    resized = cv2.resize(image, (nw, nh))
    canvas = 128 * np.ones((size, size, 3), dtype=np.uint8)
    top, left = (size - nh) // 2, (size - nw) // 2
    canvas[top:top + nh, left:left + nw] = resized
    return canvas, scale, top, left

staged_stems = []
next_numeric_id = 1

for img_path, coco_api, img_id in tqdm(downloaded, desc="Preprocessing"):
    image = cv2.imread(img_path)
    if image is None:
        continue
    orig_h, orig_w = image.shape[:2]

    # annotation parsing via the COCO API
    ann_ids = coco_api.getAnnIds(imgIds=img_id, iscrowd=False)
    anns = coco_api.loadAnns(ann_ids)
    if not anns:
        continue

    resized, scale, pad_top, pad_left = letterbox_resize(image, IMG_SIZE)

    stem = f"{next_numeric_id:012d}"
    next_numeric_id += 1

    out_img_path = os.path.join(STAGE_DIR, "images", stem + ".jpg")
    cv2.imwrite(out_img_path, resized)

    out_lines = []
    for ann in anns:
        yolo_class_id = COCO_ID_TO_YOLO_ID[ann["category_id"]]
        x, y, box_w, box_h = ann["bbox"]

        # Map box coordinates through the same resize + padding applied to the image
        xmin_r = x * scale + pad_left
        ymin_r = y * scale + pad_top
        xmax_r = (x + box_w) * scale + pad_left
        ymax_r = (y + box_h) * scale + pad_top

        xc = ((xmin_r + xmax_r) / 2.0) / IMG_SIZE
        yc = ((ymin_r + ymax_r) / 2.0) / IMG_SIZE
        w_n = (xmax_r - xmin_r) / IMG_SIZE
        h_n = (ymax_r - ymin_r) / IMG_SIZE

        out_lines.append(f"{yolo_class_id} {xc:.6f} {yc:.6f} {w_n:.6f} {h_n:.6f}")

    out_lbl_path = os.path.join(STAGE_DIR, "labels", stem + ".txt")
    with open(out_lbl_path, "w") as f:
        f.write("\n".join(out_lines))

    staged_stems.append(stem)

print(f"Preprocessed {len(staged_stems)} image/label pairs into '{STAGE_DIR}'.")
print(f"Example filenames: {staged_stems[:3]} ... (purely numeric, matches val.py's id parsing)")


Preprocessing: 100%|██████████| 4000/4000 [00:26<00:00, 150.03it/s]

Preprocessed 4000 image/label pairs into 'coco_yolo_staging'.
Example filenames: ['000000000001', '000000000002', '000000000003'] ... (purely numeric, matches val.py's id parsing)


In [10]:
# 1.6 Split the combined pool ourselves into train / val / test = 80% / 10% / 10%
random.seed(42)
random.shuffle(staged_stems)

n_total = len(staged_stems)
n_train = int(0.80 * n_total)
n_val   = int(0.10 * n_total)
# remainder -> test (keeps the split exact even with integer rounding)

train_stems = staged_stems[:n_train]
val_stems   = staged_stems[n_train:n_train + n_val]
test_stems  = staged_stems[n_train + n_val:]

print(f"Train: {len(train_stems)}  ({len(train_stems)/n_total:.1%})")
print(f"Val  : {len(val_stems)}  ({len(val_stems)/n_total:.1%})")
print(f"Test : {len(test_stems)}  ({len(test_stems)/n_total:.1%})")


Train: 3200  (80.0%)
Val  : 400  (10.0%)
Test : 400  (10.0%)


In [11]:
# 1.7 Materialize the split into the folder layout YOLOv5 expects
import shutil

FINAL_DIR = "coco_yolo_dataset"

def build_split(split_name, stems):
    img_out = os.path.join(FINAL_DIR, "images", split_name)
    lbl_out = os.path.join(FINAL_DIR, "labels", split_name)
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)
    for stem in stems:
        shutil.copy(os.path.join(STAGE_DIR, "images", stem + ".jpg"), os.path.join(img_out, stem + ".jpg"))
        shutil.copy(os.path.join(STAGE_DIR, "labels", stem + ".txt"), os.path.join(lbl_out, stem + ".txt"))

for split_name, stems in [("train", train_stems), ("val", val_stems), ("test", test_stems)]:
    build_split(split_name, stems)

print("Final YOLOv5 dataset directory structure:")
!find {FINAL_DIR} -maxdepth 2 -type d


Final YOLOv5 dataset directory structure:
coco_yolo_dataset
coco_yolo_dataset/labels
coco_yolo_dataset/labels/test
coco_yolo_dataset/labels/val
coco_yolo_dataset/labels/train
coco_yolo_dataset/images
coco_yolo_dataset/images/test
coco_yolo_dataset/images/val
coco_yolo_dataset/images/train


In [12]:
# 1.8 Write the data.yaml config YOLOv5 needs (paths + class names)
import yaml

data_yaml = {
    "path": os.path.abspath(FINAL_DIR),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": len(COCO_CLASSES),
    "names": COCO_CLASSES,
}

with open("coco_subset.yaml", "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(open("coco_subset.yaml").read()[:400], "...")


path: /content/coco_yolo_dataset
train: images/train
val: images/val
test: images/test
nc: 80
names:
- person
- bicycle
- car
- motorcycle
- airplane
- bus
- train
- truck
- boat
- traffic light
- fire hydrant
- stop sign
- parking meter
- bench
- bird
- cat
- dog
- horse
- sheep
- cow
- elephant
- bear
- zebra
- giraffe
- backpack
- umbrella
- handbag
- tie
- suitcase
- frisbee
- skis
- snowboard ...


## 2. Model Architecture
### Requirement 2: YOLOv5 base model pre-trained on COCO, fine-tuned; compare s/m/l variants


In [13]:
# 2.1 Load each pre-trained (on COCO) variant via torch.hub and inspect
import torch

YOLO_VARIANTS = ["yolov5s", "yolov5m", "yolov5l"]

variant_params = {}
for variant in YOLO_VARIANTS:
    model = torch.hub.load("ultralytics/yolov5", variant, pretrained=True, verbose=False)
    n_params = sum(p.numel() for p in model.model.parameters())
    variant_params[variant] = n_params
    print(f"{variant:10s} -> {n_params/1e6:.2f}M parameters (pre-trained on full COCO)")
    del model


The repository ultralytics_yolov5 does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /root/.cache/torch/hub/master.zip
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


YOLOv5 🚀 2026-7-21 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

100%|██████████| 14.1M/14.1M [00:00<00:00, 254MB/s]

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


yolov5s    -> 7.23M parameters (pre-trained on full COCO)


YOLOv5 🚀 2026-7-21 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

100%|██████████| 40.8M/40.8M [00:01<00:00, 40.2MB/s]

Fusing layers... 
YOLOv5m summary: 290 layers, 21172173 parameters, 0 gradients, 48.9 GFLOPs
Adding AutoShape... 
YOLOv5 🚀 2026-7-21 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)



yolov5m    -> 21.17M parameters (pre-trained on full COCO)


100%|██████████| 89.3M/89.3M [00:04<00:00, 22.5MB/s]

Fusing layers... 
YOLOv5l summary: 367 layers, 46533693 parameters, 0 gradients, 109.0 GFLOPs
Adding AutoShape... 


yolov5l    -> 46.53M parameters (pre-trained on full COCO)


## 3. Training and Evaluation
### Requirement 3: AdamW/SGD + CIoU loss; mAP / AP50 / AP75; augmentation + LR scheduling
### Also completes Requirement 2's "experiment with different variants"


In [14]:
# 3.1 Shared training configuration
EPOCHS    = 30
BATCH     = 16
IMG_SIZE  = 640
OPTIMIZER = "AdamW"

print(f"Training config -> epochs={EPOCHS}, batch={BATCH}, img_size={IMG_SIZE}, optimizer={OPTIMIZER}")
print(f"Variants to train: {YOLO_VARIANTS}")


Training config -> epochs=30, batch=16, img_size=640, optimizer=AdamW
Variants to train: ['yolov5s', 'yolov5m', 'yolov5l']


In [15]:
# --- 3.2 Fine-tune EACH variant (s, m, l) on our COCO subset ---

for variant in YOLO_VARIANTS:
    weights_out = f"runs_train/coco_{variant}_run/weights/best.pt"

    if os.path.exists(weights_out):
        print(f"\n===== Skipping {variant}: best.pt already exists at {weights_out} =====")
        continue

    print(f"\n===== Training {variant} =====")
    !python yolov5/train.py \
        --img {IMG_SIZE} \
        --batch {BATCH} \
        --epochs {EPOCHS} \
        --data coco_subset.yaml \
        --weights {variant}.pt \
        --optimizer {OPTIMIZER} \
        --cos-lr \
        --hyp yolov5/data/hyps/hyp.scratch-low.yaml \
        --name coco_{variant}_run \
        --project runs_train \
        --exist-ok

    # Verify training actually produced weights before moving to the next variant
    # if it didn't, stop here with a clear error rather than silently continuing to val.py later.
    if not os.path.exists(weights_out):
        raise RuntimeError(
            f"Training for {variant} did NOT produce {weights_out}. "
            f"Scroll up through this cell's output above to find the actual error "
            f"(e.g. CUDA out-of-memory, dataset path issue, or a Colab disconnect)."
        )



===== Training yolov5s =====
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: WARNING W&B disabled due to login timeout.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
train: weights=yolov5s.pt, cfg=, data=coco_subset.yaml, hyp=yolov5/data/hyps/hyp.scratch-low.yaml, epochs=30, batch_size=16, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=yolov5/data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=, multi_scale=False, single_cls=False, optimizer=AdamW, sync_bn=False, workers=8, project=runs_train, name=coco_yolov5s_run, exist_ok=True, quad=False, cos_lr=True, label_smoothing=0

In [16]:
# 3.3 Convert our own val split to a COCO-format ground-truth JSON, using the COCO API's expected schema
import json

def yolo_to_coco_gt(images_dir, labels_dir, class_names, out_path):
    images, annotations = [], []
    ann_id = 1
    for img_name in sorted(os.listdir(images_dir)):
        stem = os.path.splitext(img_name)[0]
        img_id = int(stem)
        img_path = os.path.join(images_dir, img_name)
        h, w = cv2.imread(img_path).shape[:2]
        images.append({"id": img_id, "file_name": img_name, "height": h, "width": w})

        lbl_path = os.path.join(labels_dir, stem + ".txt")
        if not os.path.exists(lbl_path):
            continue
        with open(lbl_path) as f:
            for line in f.read().strip().splitlines():
                if not line:
                    continue
                cls, xc, yc, bw, bh = map(float, line.split())
                box_w, box_h = bw * w, bh * h
                x = xc * w - box_w / 2
                y = yc * h - box_h / 2
                annotations.append({
                    "id": ann_id, "image_id": img_id, "category_id": int(cls),
                    "bbox": [x, y, box_w, box_h], "area": box_w * box_h, "iscrowd": 0,
                })
                ann_id += 1

    cats = [{"id": i, "name": n} for i, n in enumerate(class_names)]
    with open(out_path, "w") as f:
        json.dump({"images": images, "annotations": annotations, "categories": cats}, f)

yolo_to_coco_gt(
    images_dir=os.path.join(FINAL_DIR, "images", "val"),
    labels_dir=os.path.join(FINAL_DIR, "labels", "val"),
    class_names=COCO_CLASSES,
    out_path="coco_val_gt.json",
)
print("Ground-truth COCO-format val annotations written to coco_val_gt.json")


Ground-truth COCO-format val annotations written to coco_val_gt.json


In [17]:
# 3.4 Run YOLOv5's val.py for EACH trained variant (produces predictions.json per variant)
for variant in YOLO_VARIANTS:
    best_weights = f"runs_train/coco_{variant}_run/weights/best.pt"

    if not os.path.exists(best_weights):
        print(f"\n===== Skipping evaluation for {variant}: {best_weights} not found "
              f"(training did not complete for this variant) =====")
        continue

    print(f"\n===== Evaluating {variant} =====")
    !python yolov5/val.py \
        --img {IMG_SIZE} \
        --data coco_subset.yaml \
        --weights {best_weights} \
        --task val \
        --save-json \
        --project runs_val \
        --name coco_{variant}_val \
        --exist-ok



===== Evaluating yolov5s =====
val: data=coco_subset.yaml, weights=['runs_train/coco_yolov5s_run/weights/best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, max_det=300, task=val, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=True, project=runs_val, name=coco_yolov5s_val, exist_ok=True, half=False, dnn=False
YOLOv5 🚀 v7.0-519-g6be609f3 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
val: Scanning /content/coco_yolo_dataset/labels/val.cache... 400 images, 0 backgrounds, 0 corrupt: 100% 400/400 [00:00<?, ?it/s]
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% 13/13 [00:10<00:00,  1.18it/s]
                   all        400       3846      0.383       0.29      0.289      0.165
Speed: 0.3ms pre-process, 8.4ms inference, 4.4ms NMS per image

In [18]:
# 3.5 Score every variant's predictions with the COCO API (COCOeval) mAP, AP50, AP75
from pycocotools.coco import COCO as COCOgt
from pycocotools.cocoeval import COCOeval
import glob
# COCO API: load our ground truth
coco_gt = COCOgt("coco_val_gt.json")

results_table = []
for variant in YOLO_VARIANTS:
    best_weights = f"runs_train/coco_{variant}_run/weights/best.pt"
    val_dir = f"runs_val/coco_{variant}_val"

    if not os.path.exists(best_weights):
        print(f"[SKIP] {variant}: no trained weights found at {best_weights} "
              f"-- training did not complete for this variant.")
        continue

    matches = glob.glob(f"{val_dir}/*predictions.json")
    if not matches:
        print(f"[SKIP] {variant}: no predictions.json found in {val_dir} "
              f"-- val.py did not produce output for this variant.")
        continue

    pred_json = matches[0]
    coco_dt = coco_gt.loadRes(pred_json)                    # COCO API call
    coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")  # COCO API evaluator
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    mAP_50_95 = coco_eval.stats[0]   # mAP @ IoU 0.50:0.95
    AP50      = coco_eval.stats[1]   # AP  @ IoU 0.50
    AP75      = coco_eval.stats[2]   # AP  @ IoU 0.75

    results_table.append({
        "variant": variant,
        "params_M": round(variant_params[variant] / 1e6, 2),
        "mAP_0.5:0.95": round(mAP_50_95, 4),
        "AP50": round(AP50, 4),
        "AP75": round(AP75, 4),
    })

import pandas as pd

if not results_table:
    raise RuntimeError(
        "No variant produced usable results -- see the [SKIP] messages above to find the root cause "
        "(most commonly: training didn't finish, e.g. due to no GPU, a Colab disconnect, or an OOM error)."
    )

accuracy_df = pd.DataFrame(results_table)
print("\n=== Accuracy comparison across YOLOv5 variants (Deliverable 3 / Criterion 1) ===")
accuracy_df


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=2.45s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=6.64s).
Accumulating evaluation results...
DONE (t=1.63s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.165
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.288
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.166
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.083
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.156
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.239
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.197
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.354
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets

,variant,params_M,mAP_0.5:0.95,AP50,AP75
0,yolov5s,7.23,0.1645,0.2883,0.1658
1,yolov5m,21.17,0.1785,0.2964,0.1907
2,yolov5l,46.53,0.1830,0.3017,0.1977


In [19]:
# 3.6 Visualize training curves (loss, mAP over epochs) for each variant
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for variant in YOLO_VARIANTS:
    df = pd.read_csv(f"runs_train/coco_{variant}_run/results.csv")
    df.columns = [c.strip() for c in df.columns]
    axes[0].plot(df["epoch"], df["val/box_loss"], label=f"{variant} val box (CIoU) loss")
    axes[1].plot(df["epoch"], df["metrics/mAP_0.5"], label=f"{variant} mAP@0.5")

axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("CIoU Box Loss"); axes[0].legend(); axes[0].set_title("Validation Box Loss")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("mAP@0.5"); axes[1].legend(); axes[1].set_title("Validation mAP@0.5")
plt.tight_layout()
plt.show()


In [20]:
# 3.7 Pick the best variant (by mAP@0.5:0.95) to carry forward into deployment sections
best_row = accuracy_df.sort_values("mAP_0.5:0.95", ascending=False).iloc[0]
VARIANT = best_row["variant"]
BEST_WEIGHTS = f"runs_train/coco_{VARIANT}_run/weights/best.pt"

print(f"Best-performing variant: {VARIANT}  (mAP@0.5:0.95 = {best_row['mAP_0.5:0.95']})")
print(f"Carrying forward weights: {BEST_WEIGHTS}")


Best-performing variant: yolov5l  (mAP@0.5:0.95 = 0.183)
Carrying forward weights: runs_train/coco_yolov5l_run/weights/best.pt


## 4. Real-Time Object Detection System
### Requirement 4: OpenCV + PyTorch inference on live video / video files / images
### Criterion 2: Speed (inference time, FPS)


In [22]:
# 4.1 Load the best fine-tuned model for inference
import torch, cv2, time

detector = torch.hub.load("ultralytics/yolov5", "custom", path=BEST_WEIGHTS, force_reload=False)
detector.conf = 0.35   # confidence threshold
detector.iou  = 0.45   # NMS IoU threshold

device = "cuda" if torch.cuda.is_available() else "cpu"
detector.to(device)
print(f"Detector ({VARIANT}) loaded on: {device}")


Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2026-7-21 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 267 layers, 46533693 parameters, 0 gradients, 109.0 GFLOPs
Adding AutoShape... 


Detector (yolov5l) loaded on: cuda


In [23]:
# 4.2 Single-image inference
def detect_image(image_path, save_path="detection_output.jpg"):
    frame = cv2.imread(image_path)
    results = detector(frame[:, :, ::-1])         # BGR -> RGB
    annotated = results.render()[0][:, :, ::-1]   # RGB -> BGR for OpenCV/imwrite
    cv2.imwrite(save_path, annotated)
    print(f"Saved annotated detection to {save_path}")
    return annotated

# Example (pick a real image from the held-out test split):
test_dir = os.path.join(FINAL_DIR, "images", "test")
if os.listdir(test_dir):
    sample_img = os.path.join(test_dir, os.listdir(test_dir)[0])
    detect_image(sample_img)


Saved annotated detection to detection_output.jpg


In [32]:
# Then point detect_video() at your file, e.g.:
detect_video("/persons.mp4", save_path="detection_output.mp4")

Processed video saved to detection_output.mp4. Average inference FPS: 37.14


37.14253691894738

In [35]:
# 4.3 Video-file inference with per-frame timing
def detect_video(video_path, save_path="detection_output.mp4"):
    cap = cv2.VideoCapture(video_path)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    fps_in = cap.get(cv2.CAP_PROP_FPS) or 25
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(save_path, fourcc, fps_in, (w, h))

    frame_times = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        t0 = time.time()
        results = detector(frame[:, :, ::-1])
        annotated = results.render()[0][:, :, ::-1]
        frame_times.append(time.time() - t0)
        writer.write(annotated)

    cap.release()
    writer.release()
    avg_fps = 1.0 / (sum(frame_times) / len(frame_times)) if frame_times else 0
    print(f"Processed video saved to {save_path}. Average inference FPS: {avg_fps:.2f}")
    return avg_fps


In [25]:
# 4.4 Benchmark inference SPEED across all three trained variants (Criterion 2: Speed)
speed_rows = []
n_warmup, n_timed = 5, 30
test_images = [os.path.join(test_dir, f) for f in os.listdir(test_dir)[:n_timed]]

for variant in YOLO_VARIANTS:
    weights_path = f"runs_train/coco_{variant}_run/weights/best.pt"
    m = torch.hub.load("ultralytics/yolov5", "custom", path=weights_path, force_reload=False)
    m.to(device)

    for img_path in test_images[:n_warmup]:
        frame = cv2.imread(img_path)
        _ = m(frame[:, :, ::-1])

    times = []
    for img_path in test_images:
        frame = cv2.imread(img_path)
        t0 = time.time()
        _ = m(frame[:, :, ::-1])
        times.append(time.time() - t0)

    avg_ms = (sum(times) / len(times)) * 1000
    fps = 1000 / avg_ms
    speed_rows.append({"variant": variant, "avg_inference_ms": round(avg_ms, 2), "fps": round(fps, 2)})
    del m

speed_df = pd.DataFrame(speed_rows)
print("\n=== Speed comparison across YOLOv5 variants (Criterion 2) ===")
speed_df


Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2026-7-21 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 
Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2026-7-21 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 212 layers, 21172173 parameters, 0 gradients, 48.9 GFLOPs
Adding AutoShape... 
Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2026-7-21 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 267 layers, 46533693 parameters, 0 gradients, 109.0 GFLOPs
Adding AutoShape... 



=== Speed comparison across YOLOv5 variants (Criterion 2) ===


,variant,avg_inference_ms,fps
0,yolov5s,21.07,47.46
1,yolov5m,23.54,42.48
2,yolov5l,38.39,26.05


## 5. Model Deployment
### Requirement 5: Deploy on edge devices (Jetson/RPi), optimize with quantization + pruning


In [26]:
# 5.1 Export best-variant weights to ONNX and TorchScript for edge deployment
!python yolov5/export.py \
    --weights {BEST_WEIGHTS} \
    --img {IMG_SIZE} \
    --include onnx torchscript \
    --simplify


export: data=yolov5/data/coco128.yaml, weights=['runs_train/coco_yolov5l_run/weights/best.pt'], imgsz=[640], batch_size=1, device=cpu, half=False, inplace=False, keras=False, optimize=False, int8=False, per_tensor=False, dynamic=False, cache=, simplify=True, mlmodel=False, opset=17, verbose=False, workspace=4, nms=False, agnostic_nms=False, topk_per_class=100, topk_all=100, iou_thres=0.45, conf_thres=0.25, include=['onnx', 'torchscript']
YOLOv5 🚀 v7.0-519-g6be609f3 Python-3.12.13 torch-2.11.0+cu128 CPU

Fusing layers... 
Model summary: 267 layers, 46533693 parameters, 0 gradients, 109.0 GFLOPs

PyTorch: starting from runs_train/coco_yolov5l_run/weights/best.pt with output shape (1, 25200, 85) (89.4 MB)

TorchScript: starting export with torch 2.11.0+cu128...
TorchScript: export success ✅ 14.4s, saved as runs_train/coco_yolov5l_run/weights/best.torchscript (178.2 MB)
requirements: Ultralytics requirement ['onnxscript'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment

In [27]:
# 5.2 Dynamic quantization (INT8 weights) for faster CPU inference, e.g. Raspberry Pi
import torch.nn as nn

fp32_model = torch.hub.load("ultralytics/yolov5", "custom", path=BEST_WEIGHTS, force_reload=False)
fp32_model.eval()

quantized_model = torch.quantization.quantize_dynamic(
    fp32_model.model,
    {nn.Linear, nn.Conv2d},
    dtype=torch.qint8,
)
torch.save(quantized_model.state_dict(), "yolov5_quantized.pt")

orig_size  = os.path.getsize(BEST_WEIGHTS) / 1e6
quant_size = os.path.getsize("yolov5_quantized.pt") / 1e6
print(f"Original weights size : {orig_size:.2f} MB")
print(f"Quantized weights size: {quant_size:.2f} MB")


Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2026-7-21 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 267 layers, 46533693 parameters, 0 gradients, 109.0 GFLOPs
Adding AutoShape... 
/tmp/ipykernel_680/4112004806.py:7: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.qu

Original weights size : 93.70 MB
Quantized weights size: 186.22 MB


In [28]:
# 5.3 Magnitude pruning: zero out the smallest-magnitude conv weights
import torch.nn.utils.prune as prune

prune_model = torch.hub.load("ultralytics/yolov5", "custom", path=BEST_WEIGHTS, force_reload=False)
prune_model.eval()

PRUNE_AMOUNT = 0.30  # remove 30% of weights per conv layer, ranked by L1 magnitude

for module in prune_model.model.modules():
    if isinstance(module, nn.Conv2d):
        prune.l1_unstructured(module, name="weight", amount=PRUNE_AMOUNT)
        prune.remove(module, "weight")

torch.save(prune_model.model.state_dict(), "yolov5_pruned.pt")
print(f"Pruned {PRUNE_AMOUNT:.0%} of conv weights (L1 magnitude) and saved to yolov5_pruned.pt")


Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2026-7-21 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 267 layers, 46533693 parameters, 0 gradients, 109.0 GFLOPs
Adding AutoShape... 


Pruned 30% of conv weights (L1 magnitude) and saved to yolov5_pruned.pt


## 6. Robustness Testing
### Criterion 3: Robustness to lighting conditions, occlusions, and object variations

We synthetically perturb a held-out sample of test images in three controlled ways and
re-run detection, comparing detection **count** and **average confidence** against the
unperturbed originals. This directly measures how much performance degrades under each
condition, which is what "robustness" means in the evaluation criteria.


In [29]:
# 6.1 Define the three perturbation types
def apply_lighting_change(image, factor):
    """factor > 1 brightens, factor < 1 darkens (simulates different lighting conditions)."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

def apply_occlusion(image, patch_fraction=0.25):
    """Blacks out a random square patch to simulate a partially occluded object."""
    out = image.copy()
    h, w = out.shape[:2]
    patch_h, patch_w = int(h * patch_fraction), int(w * patch_fraction)
    y0 = random.randint(0, h - patch_h)
    x0 = random.randint(0, w - patch_w)
    out[y0:y0 + patch_h, x0:x0 + patch_w] = 0
    return out

def apply_scale_variation(image, scale_factor):
    """Shrinks the image then pads back to original size (simulates smaller/farther objects)."""
    h, w = image.shape[:2]
    new_h, new_w = int(h * scale_factor), int(w * scale_factor)
    small = cv2.resize(image, (new_w, new_h))
    canvas = np.zeros_like(image)
    top, left = (h - new_h) // 2, (w - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = small
    return canvas

print("Perturbation functions ready: lighting, occlusion, scale variation.")


Perturbation functions ready: lighting, occlusion, scale variation.


In [30]:
# 6.2 Run the robustness benchmark: baseline vs. each perturbation
ROBUSTNESS_SAMPLE = [os.path.join(test_dir, f) for f in os.listdir(test_dir)[:40]]

def score_condition(image_list, transform_fn=None):
    """Runs detection over a list of images and returns (avg_num_detections, avg_confidence)."""
    detection_counts, confidences = [], []
    for img_path in image_list:
        frame = cv2.imread(img_path)
        if frame is None:
            continue
        if transform_fn is not None:
            frame = transform_fn(frame)
        results = detector(frame[:, :, ::-1])
        preds = results.xyxy[0]  # tensor: [x1, y1, x2, y2, conf, cls]
        detection_counts.append(len(preds))
        if len(preds) > 0:
            confidences.append(preds[:, 4].mean().item())
    avg_count = np.mean(detection_counts) if detection_counts else 0
    avg_conf = np.mean(confidences) if confidences else 0
    return avg_count, avg_conf

conditions = {
    "Baseline (no perturbation)": None,
    "Low light (0.4x brightness)": lambda im: apply_lighting_change(im, 0.4),
    "Bright light (1.8x brightness)": lambda im: apply_lighting_change(im, 1.8),
    "Occlusion (25% random patch)": lambda im: apply_occlusion(im, 0.25),
    "Scale variation (50% smaller)": lambda im: apply_scale_variation(im, 0.5),
}

robustness_rows = []
for cond_name, fn in conditions.items():
    avg_count, avg_conf = score_condition(ROBUSTNESS_SAMPLE, fn)
    robustness_rows.append({
        "condition": cond_name,
        "avg_detections_per_image": round(avg_count, 2),
        "avg_confidence": round(avg_conf, 3),
    })

robustness_df = pd.DataFrame(robustness_rows)
print("\n=== Robustness comparison (Criterion 3) ===")
robustness_df



=== Robustness comparison (Criterion 3) ===


,condition,avg_detections_per_image,avg_confidence
0,Baseline (no perturbation),4.82,0.561
1,Low light (0.4x brightness),4.18,0.560
2,Bright light (1.8x brightness),4.28,0.572
3,Occlusion (25% random patch),4.82,0.546
4,Scale variation (50% smaller),3.28,0.592


In [31]:
# 6.3 Visualize robustness degradation relative to baseline
baseline_count = robustness_df.iloc[0]["avg_detections_per_image"]
robustness_df["pct_of_baseline_detections"] = (
    robustness_df["avg_detections_per_image"] / baseline_count * 100
).round(1)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(robustness_df["condition"], robustness_df["pct_of_baseline_detections"], color="steelblue")
ax.axhline(100, color="gray", linestyle="--", linewidth=1)
ax.set_ylabel("Detections retained vs. baseline (%)")
ax.set_title("Robustness: detection retention under perturbation")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

robustness_df


,condition,avg_detections_per_image,avg_confidence,pct_of_baseline_detections
0,Baseline (no perturbation),4.82,0.561,100.0
1,Low light (0.4x brightness),4.18,0.560,86.7
2,Bright light (1.8x brightness),4.28,0.572,88.8
3,Occlusion (25% random patch),4.82,0.546,100.0
4,Scale variation (50% smaller),3.28,0.592,68.0
